## Download climate data from the HARMONIZE STAC
<hr style="border:1px solid #274ad4;">

In [2]:
# Installing the libraries required to access the STAC catalog and process the data
import importlib

packages = {
    "ipykernel": "ipykernel",
    "pystac_client": "pystac-client",
    "geopandas": "geopandas",
    "pandas": "pandas",
    "tqdm": "tqdm"
}

for module, pkg in packages.items():
    try:
        importlib.import_module(module)
        print(f"OK: {module}")
    except ImportError:
        !pip install -U {pkg}
        print(f"INSTALADO: {module}")

OK: ipykernel
OK: pystac_client
OK: geopandas
OK: pandas
OK: tqdm


In [3]:
# Importing libraries
import os
import pandas as pd
import geopandas as gpd
from pystac_client import Client
from tqdm import tqdm

### 1. Set constants

In [4]:
# HARMONIZE STAC endpoint and period of interest
HARMONIZE_STAC_URL = "https://geolab.inpe.br/bdc/harmonize/stac/v1/"

START_DATE = "2010-01-01"
END_DATE = "2025-12-31"

### 2. List HARMONIZE STAC collections

In [5]:
# Connect to the HARMONIZE STAC catalog and list the available data collections
catalog = Client.open(HARMONIZE_STAC_URL)
catalog.add_conforms_to("ITEM_SEARCH")  # ensures item search is enabled for this endpoint

print("Available collections:")

for collection in catalog.get_collections():
    print(collection.id)

Available collections:
precip_mean_no_mun_epiweek_era5land-1
precip_max_no_mun_month_cptec-1
precip_max_ne_mun_epiweek_era5land-1
chikungunya_cases_notified_NO_mun_epiweek_infodengue-1
temp_mean_ne_mun_month_cptec-1
precip_max_no_mun_epiweek_cptec-1
precip_max_no_mun_epiweek_era5land-1
precip_mean_ne_mun_epiweek_cptec-1
humidity_percent_ne_mun_epiweek_era5land-1
chikungunya_alert_level_NE_mun_epiweek_infodengue-1
precip_min_ne_mun_epiweek_cptec-1
precip_mean_no_mun_month_cptec-1
anomaly_cdays_temp_no_mun_month_era5land-1
temp_mean_ne_mun_month_era5land-1
anomaly_cdays_temp_ne_mun_month_era5land-1
precip_min_no_mun_epiweek_cptec-1
temp_mean_no_mun_epiweek_cptec-1
precip_max_ne_mun_epiweek_cptec-1
precip_min_ne_mun_epiweek_era5land-1
temp_max_ne_mun_epiweek_cptec-1
chikungunya_cases_notified_NE_mun_epiweek_infodengue-1
humidity_percent_no_mun_epiweek_era5land-1
precip_mean_no_mun_month_era5land-1
chikungunya_incidence_NO_mun_epiweek_infodengue-1
zika_alert_level_NE_mun_epiweek_infodengue

### 3. Obtain climate data for each municipality

In [6]:
# 3.1 Set collection id (temperature, precipitation, humidity, etc.)

# COLLECTION_ID = "precip_max_no_mun_epiweek_era5land-1"
# COLLECTION_ID = "humidity_percent_no_mun_epiweek_era5land-1"
COLLECTION_ID = "temp_mean_no_mun_epiweek_era5land-1"

In [7]:
# 3.2 Search climate data on the HARMONIZE STAC
stac_search = catalog.search(
    collections=[COLLECTION_ID],
    datetime=f"{START_DATE}/{END_DATE}",
    limit=1000
)

items = list(stac_search.items())

print("Items found:", len(items))

Items found: 835


In [9]:
# 3.3 For each item, retrieve the GeoJSON asset as a GeoDataFrame
def item_to_geodataframe(item):
    datetime = item.properties["datetime"]
    geojson_href = item.assets["geojson"].href

    gdf = gpd.read_file(geojson_href)
    gdf["year_month"] = pd.to_datetime(datetime).strftime("%Y-%m")

    return gdf

gdfs = [item_to_geodataframe(item) for item in tqdm(items)]

# Display the first two GeoDataFrames
for gdf in gdfs[:2]:
    display(gdf.head())

100%|██████████| 835/835 [15:55<00:00,  1.14s/it]


,cod_mun,name_mun,uf_mun,data_source,name_indicator,epiweek_number,epiweek_start_date,time_agg,spatial_agg,value,geometry,year_month
0,1500107,Abaetetuba,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.184008,"POLYGON ((-48.83139 -1.56352, -48.8291 -1.5710...",2025-12
1,1500800,Ananindeua,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.327421,"POLYGON ((-48.33466 -1.23983, -48.33379 -1.243...",2025-12
2,1501105,Bagre,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.197905,"POLYGON ((-50.29255 -1.80634, -50.28356 -1.806...",2025-12
3,1501204,Baião,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.601133,"POLYGON ((-49.48643 -2.66318, -49.43872 -2.663...",2025-12
4,1501303,Barcarena,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.270811,"POLYGON ((-48.4695 -1.59545, -48.46545 -1.6021...",2025-12


,cod_mun,name_mun,uf_mun,data_source,name_indicator,epiweek_number,epiweek_start_date,time_agg,spatial_agg,value,geometry,year_month
0,1500107,Abaetetuba,PA,ERA5-Land (Copernicus),temp_mean,52,2025-12-21,epiweek,municipality,25.617170,"POLYGON ((-48.83139 -1.56352, -48.8291 -1.5710...",2025-12
1,1500800,Ananindeua,PA,ERA5-Land (Copernicus),temp_mean,52,2025-12-21,epiweek,municipality,25.924774,"POLYGON ((-48.33466 -1.23983, -48.33379 -1.243...",2025-12
2,1501105,Bagre,PA,ERA5-Land (Copernicus),temp_mean,52,2025-12-21,epiweek,municipality,25.468920,"POLYGON ((-50.29255 -1.80634, -50.28356 -1.806...",2025-12
3,1501204,Baião,PA,ERA5-Land (Copernicus),temp_mean,52,2025-12-21,epiweek,municipality,25.586357,"POLYGON ((-49.48643 -2.66318, -49.43872 -2.663...",2025-12
4,1501303,Barcarena,PA,ERA5-Land (Copernicus),temp_mean,52,2025-12-21,epiweek,municipality,25.761169,"POLYGON ((-48.4695 -1.59545, -48.46545 -1.6021...",2025-12


In [10]:
# 3.4 Concatenate the list of GeoDataFrames into one DataFrame and remove the geometry column
df_all = pd.concat(gdfs, ignore_index=True)

final_df = pd.DataFrame(df_all.drop(columns="geometry")).copy()

print(final_df.shape)
final_df.head()

(17535, 11)


,cod_mun,name_mun,uf_mun,data_source,name_indicator,epiweek_number,epiweek_start_date,time_agg,spatial_agg,value,year_month
0,1500107,Abaetetuba,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.184008,2025-12
1,1500800,Ananindeua,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.327421,2025-12
2,1501105,Bagre,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.197905,2025-12
3,1501204,Baião,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.601133,2025-12
4,1501303,Barcarena,PA,ERA5-Land (Copernicus),temp_mean,53,2025-12-28,epiweek,municipality,27.270811,2025-12


In [11]:
# Display the column names available in the dataset
final_df.columns

Index(['cod_mun', 'name_mun', 'uf_mun', 'data_source', 'name_indicator',
       'epiweek_number', 'epiweek_start_date', 'time_agg', 'spatial_agg',
       'value', 'year_month'],
      dtype='str')

### 4. Save data

In [12]:
# Save the final DataFrame as a CSV file named after the collection
FINAL_CSV_NAME = f"{COLLECTION_ID}.csv"

final_df.to_csv(FINAL_CSV_NAME, sep=",", decimal=".", index=False, header=True)

# Show the directory where the file was saved
print(os.path.join(os.getcwd(), FINAL_CSV_NAME))

c:\Users\Administrador\Dropbox\github_projects\application-scenario-paper\source\temp_mean_no_mun_epiweek_era5land-1.csv
